In [1]:
# manual hyperparameter tuning is here! production ready!
!pip install quick-sentiments==0.6.4


   ---------------------------------------- 0.0/2.6 MB ? eta -:--:--
   ----------------------------------- ---- 2.4/2.6 MB 13.8 MB/s eta 0:00:01
   ---------------------------------------- 2.6/2.6 MB 12.1 MB/s  0:00:00
  Attempting uninstall: quick-sentiments
    Found existing installation: quick-sentiments 0.6.3
    Uninstalling quick-sentiments-0.6.3:
      Successfully uninstalled quick-sentiments-0.6.3


In [2]:
!pip show quick_sentiments #check for latest version

Name: quick-sentiments
Version: 0.6.4
Summary: Sentiment Analysis pipeline
Home-page: https://github.com/AlabhyaMe/Sentiments-Analysis
Author: Alabhya Dahal
Author-email: Alabhya Dahal <alabhya.dahal@gmail.com>
License: MIT License
Location: C:\Users\meala\anaconda3\envs\quicksentiment\Lib\site-packages
Requires: gensim, nltk, numpy, pandas, polars, scikit-learn, scipy, spacy, xgboost
Required-by: 


In [3]:
import polars as pl

# here I have three python script I built to pre_process the data and running the pipeline
# you can find the code in the tools/preprocess.py file
# you can find  the code in the tools/pipeline.py file
# the pre_process function is used to clean the text data, there are various options available, please check the tools/preprocess.py file for details
# the run_pipeline function is used to run the sentimental analysis pipeline, it takes the training data and the vectorizer and machine learning methods as input, and returns the results
from quick_sentiments import pre_process_nltk
from quick_sentiments import pre_process_spacy
from quick_sentiments import run_pipeline
from quick_sentiments import make_predictions
from quick_sentiments import evaluate_performance

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\meala\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\meala\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


### Training Dataset


In [4]:
# this template uses csv files 
# column names can be set in Python but this template does not automatically update the column for the demo 
# however, the function will give you the option to tell column names for the text and label data

df_train = pl.read_csv("demo/training_data/train.csv",encoding='ISO-8859-1') 
print(f"Dataset shape: {df_train.shape[0]} rows and {df_train.shape[1]} columns")


Dataset shape: 162758 rows and 5 columns


### DEMO

In [5]:
df_train.head()
# randomly select only 10% of the data since the dataset is large
#RUN ONLY ONCE
df_train = df_train.sample(fraction=0.05, shuffle=True, seed=42) 
df_train.head(5)

movieid,reviewerName,isFrequentReviewer,reviewText,sentiment
str,str,bool,str,str
"""don_vito_corleone_willy_wonka_…","""Jacob Hansen Jr.""",true,"""An acceptably mindless sanctua…","""NEGATIVE"""
"""annie_hall_james_t._kirk_hiccu…","""Manuel Ramirez""",false,"""Although many shots are out of…","""POSITIVE"""
"""ellis_redding_legend_forrest_g…","""Jose Mccormick""",false,null,"""NEGATIVE"""
"""infinite_gandalf_the_grey""","""Heidi Wood""",false,"""10 Cloverfield Lane is an exci…","""POSITIVE"""
"""katniss_everdeen_darth_vader_n…","""Nicholas Park""",true,"""It's funny, fast, and charming…","""POSITIVE"""


The dataset is for training. The sentiments are already labeled. This will allow us to train a model that can predict sentiments on new data.


In [6]:
# you can use the pre_process function to clean the text data
response_column = "reviewText" # this is the column name for the text data, feel free to change it to your text column name
sentiment_column = "sentiment" # this is the column name for the sentiment data, feel free to change it to your sentiment column name


In [7]:
# make changes as necessary
# inside the map_elements, add  the parameters [pre_process(x, parameters_to_be_added)] and set it True/False if it differs from the defualt value
# check the tools/preprocess.py file for the parameters and their default values
# some of the parameters are remove_brackets, remove_stopwords, remove_punctuation, remove_numbers, remove_emojis, remove_urls, remove_html_tags, lemmatize, stem, lowercase
df_train = pre_process_nltk(df_train, text_column=response_column, new_column_name="cleaned_text_nltk")


In [8]:
df_train = pre_process_spacy(df_train, text_column=response_column, new_column_name="cleaned_text_spacy")

In [9]:
df_train.head()

movieid,reviewerName,isFrequentReviewer,reviewText,sentiment,cleaned_text_nltk,cleaned_text_spacy
str,str,bool,str,str,str,str
"""don_vito_corleone_willy_wonka_…","""Jacob Hansen Jr.""",true,"""An acceptably mindless sanctua…","""NEGATIVE""","""an acceptably mindless sanctua…","""an acceptably mindless sanctua…"
"""annie_hall_james_t._kirk_hiccu…","""Manuel Ramirez""",false,"""Although many shots are out of…","""POSITIVE""","""although many shots are out of…","""although many shots are out of…"
"""ellis_redding_legend_forrest_g…","""Jose Mccormick""",false,null,"""NEGATIVE""","""""",""""""
"""infinite_gandalf_the_grey""","""Heidi Wood""",false,"""10 Cloverfield Lane is an exci…","""POSITIVE""","""cloverfield lane is an excitin…","""cloverfield lane is an excitin…"
"""katniss_everdeen_darth_vader_n…","""Nicholas Park""",true,"""It's funny, fast, and charming…","""POSITIVE""","""it s funny fast and charming""","""it s funny fast and charming"""


In [ ]:
#### 6 plus text representation / vectorizer methods available 
#### in the function run_pipeline (in python cell below), we shall make use of this, write the words inside [ ] for the methods you want to use
#### 1. Bag of Words [BOW] 
#### 2. Term Frequency [tf]
#### 3. TF -IDF    [tfidf]
#### 4. Word Embedding using Word2Vec (you can use other packages with slight changes) [wv] 
         # Word Embedding uses defualt 300 values; this will take some time to run
#### 5. Glove (you can use other packages with slight changes) [glove_25,glove_50, glove_100, gl0ve_200]
#### 6. Hugging Face Transformers (you can use other packages with slight changes) [transformer]. You have to download hugging face transformer models to use this method, check the tools/pipeline.py file for details

In [ ]:
#### 6 there are also five machine learning methods that can be used
#### 1. Logistic Regression [logit]
#### 2. Random forest (recommended) (rf)
#### 3. XGBoosting  [XGB](word embedding and XGBoost may take long time to complete, combination of both is not recommended in local machine)
#### 4. Naive Bayes [nb]
#### 5. Neural Network [nn] (this will take some time to run, and may run out of memory if the dataset is large, so be careful when using this method)
#### 6. Tensorflow/Keras [tf, tensorflow, keras] (this will take some time to run, and may run out of memory if the dataset is large, so be careful when using this method)


In [10]:
# example parameters for all
#Logit
custom_logit = {
    'C': [0.01, 0.1, 1.0, 10.0, 100.0],
    'solver': ['liblinear', 'lbfgs', 'saga'],
    'class_weight': [None, 'balanced'],
    'max_iter': [500, 1000, 2000]
}
#Random Forest
custom_rf = {
    'n_estimators': [100, 300, 500],
    'max_depth': [10, 20, 50, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'bootstrap': [True, False],
    'class_weight': [None, 'balanced', 'balanced_subsample']
}
#XGBoost
custom_xgb = {
    'n_estimators': [100, 300, 500],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 5, 7, 9],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'gamma': [0, 0.1, 0.2] # Minimum loss reduction required to make a further partition
}
#Neural Network
custom_nn = {
    'hidden_layer_sizes': [(50,), (100,), (50, 50), (100, 50)],
    'activation': ['relu', 'tanh', 'logistic'],
    'solver': ['adam', 'sgd'],
    'alpha': [0.0001, 0.001, 0.01, 0.1],
    'learning_rate_init': [0.001, 0.01] 
}
#Tensorflow/Keras
custom_tf = {
    'model__hidden_units': [64, 128, 256],
    'model__dropout_rate': [0.2, 0.3, 0.5],
    'batch_size': [32, 64, 128],
    'epochs': [10, 20] 
}
#Naive Bayers Multinomial
custom_nb_multinomial = {
    'alpha': [0.001, 0.01, 0.1, 0.5, 1.0, 5.0, 10.0], # Smoothing parameter
    'fit_prior': [True, False]
}
#Naive Bayes Gaussian (If using Word2Vec, GloVe, or Hugging Face (Gaussian)
custom_nb_gaussian = {
    'var_smoothing': [1e-9, 1e-8, 1e-7, 1e-6, 1e-5] # Portion of the largest variance of all features
}


In [16]:
# this is the example of how to use the function
# you can change the vectorizer_name and model_name to the ones you want to use
# for now we will use word embedding and tensorflow
# write the name of your columns in the text_column_name and sentiment_column_name
# the text_column_name is the column name of the text data, and sentiment_column_name is

# run_pipeline function will return the dataframe with the vectorized text, vectorizer used  and the model
# it will also print the results of the model, including the accuracy and F1 score
# note, even without hyperparameter tuning, the model is getting over 70% accuracy in my test
# there may not be a need to perform hyperparameter tuning, but you can set perform_tuning to True if you want to do that
model = run_pipeline(
    # Primary parameters that are required
    vectorizer_name="BOW", # BOW, tf, tfidf, wv,  glove_25,glove_50, glove_100, gl0ve_200,
    model_name="logit", # logit, rf, XGB, nb, nn, tf for tensorflow models .#XGB takes long time, can not recommend using it on normal case
    df=df_train,
    text_column_name="cleaned_text_spacy",  # this is the column name of the text data, 
    sentiment_column_name = "sentiment",
    
    # Optional parameters, 
    perform_tuning = True, # make this true if you want to perform hyperparameter tuning, it will take longer time and 
    param_grid=custom_logit,
    interactive=False, 
    random_state=259
)

   - Interactive mode disabled. Fail-fast safety engaged.
--- Running Pipeline for Bow + Logit ---
1. Splitting data into train/test...
2. Vectorizing  dataset (X)...
   - Generating Bag-of-Words features...
   - Transforming test data using fitted Bag-of-Words vectorizer...
3. Training and predicting...
   - Starting Logistic Regression training with CUSTOM PARAMETER, GridSearchCV for hyperparameter tuning...
Fitting 5 folds for each of 90 candidates, totalling 450 fits

   - Best Hyperparameters found:
{'C': 1.0, 'class_weight': 'balanced', 'max_iter': 500, 'solver': 'lbfgs'}
   - Best Cross-Validation Score (F1-weighted): 0.7202

Best model parameters: {'C': 1.0, 'class_weight': 'balanced', 'dual': False, 'fit_intercept': True, 'intercept_scaling': 1, 'l1_ratio': 0.0, 'max_iter': 500, 'n_jobs': None, 'penalty': 'deprecated', 'random_state': 259, 'solver': 'lbfgs', 'tol': 0.0001, 'verbose': 0, 'warm_start': False}
4. Evaluating model...

Classification Report:
              precision

In [17]:
evaluate_performance(model["y_test"], model["y_prob_matrix"][:, 1],positive_label=1) #alphanumeric labels, 0 is negative, 1 is positive in this case, change it as necessary

{'best_roc_threshold': np.float64(0.6242),
 'best_pr_threshold': np.float64(0.2545),
 'decile_table':     Threshold  Accuracy  Precision  Recall     F1
 0         0.0     0.671      0.671   1.000  0.803
 1         0.1     0.703      0.700   0.976  0.815
 2         0.2     0.719      0.722   0.944  0.819
 3         0.3     0.728      0.749   0.893  0.815
 4         0.4     0.733      0.777   0.845  0.810
 5         0.5     0.727      0.800   0.789  0.795
 6         0.6     0.697      0.842   0.675  0.749
 7         0.7     0.660      0.867   0.583  0.697
 8         0.8     0.623      0.909   0.486  0.634
 9         0.9     0.527      0.926   0.321  0.476
 10        1.0     0.329      0.000   0.000  0.000}

In [20]:
evaluate_performance(model["y_test"], model["y_prob_matrix"][:, 0],positive_label=0) # negative label as reference, change it as necessary

{'best_roc_threshold': np.float64(0.3778),
 'best_pr_threshold': np.float64(0.3778),
 'decile_table':     Threshold  Accuracy  Precision  Recall     F1
 0         0.0     0.329      0.329   1.000  0.495
 1         0.1     0.527      0.406   0.948  0.569
 2         0.2     0.623      0.463   0.901  0.611
 3         0.3     0.660      0.490   0.817  0.613
 4         0.4     0.697      0.529   0.743  0.618
 5         0.5     0.727      0.583   0.599  0.591
 6         0.6     0.733      0.616   0.506  0.555
 7         0.7     0.728      0.642   0.392  0.487
 8         0.8     0.719      0.697   0.261  0.380
 9         0.9     0.703      0.752   0.147  0.246
 10        1.0     0.671      0.000   0.000  0.000}

In [21]:
## the model is a dictionary that contains the results of the model, including the accuracy and F1 score

# you can access the results using the keys of the dictionary
print("Vectorizer used: ", model["vectorizer_name"])
print("Model used: ", model["model_object"])
print("Accuracy: ", model["accuracy"])



Vectorizer used:  BOW
Model used:  LogisticRegression(class_weight='balanced', max_iter=500, random_state=259)
Accuracy:  0.7266584766584766


### New Dataset for prediction
You can use the same format as the training dataset, but ensure that it contains the "Response" column for text data. The "Sentiment" column is optional for prediction datasets, as it will be generated by the model.
Make sure the dataset is saved in the "New Data" folder and is in CSV format.

In [22]:
new_data = pl.read_csv("demo/new_data/test.csv",encoding='ISO-8859-1') #keep your file here
print(new_data.shape)
new_data= new_data.sample(fraction=0.25, shuffle=True, seed=42)
print(new_data.shape)

(55315, 4)
(13828, 4)


In [23]:
new_data = pre_process_nltk(new_data, text_column=response_column, new_column_name="cleaned_text")
new_data.head()

movieid,reviewerName,isTopCritic,reviewText,cleaned_text
str,str,bool,str,str
"""spectacular_whirlwind_dazzling…","""Craig Lambert""",false,"""Cranston commands the screen, …","""cranston commands the screen b…"
"""astonish_katniss_everdeen_myri…","""Sara French""",true,"""All good stuff. Rock delivers …","""all good stuff rock delivers s…"
"""gandalf_michael_corleone_edwar…","""Mallory Chung""",false,"""Downey Jr. continues with his …","""downey jr continues with his h…"
"""epic_stardust_hermione_granger…","""Anna Camacho""",false,"""It's not a great film, but let…","""it s not a great film but let …"
"""ellis_redding_t-800_harry_pott…","""Cheryl Fuller""",false,"""RECOMMENDED ""Deliverance"" with…","""recommended deliverance with a…"


In [24]:
make_predictions(
    new_data=new_data,
    text_column_name="cleaned_text",  # this is the column name of the text data,
    prediction_column_name="sentiment_predictions",  # Optional custom name
    trained_results=model
)

movieid,reviewerName,isTopCritic,reviewText,cleaned_text,sentiment_predictions,confidence
str,str,bool,str,str,str,f64
"""spectacular_whirlwind_dazzling…","""Craig Lambert""",false,"""Cranston commands the screen, …","""cranston commands the screen b…","""POSITIVE""",0.746572
"""astonish_katniss_everdeen_myri…","""Sara French""",true,"""All good stuff. Rock delivers …","""all good stuff rock delivers s…","""POSITIVE""",0.92632
"""gandalf_michael_corleone_edwar…","""Mallory Chung""",false,"""Downey Jr. continues with his …","""downey jr continues with his h…","""NEGATIVE""",0.652623
"""epic_stardust_hermione_granger…","""Anna Camacho""",false,"""It's not a great film, but let…","""it s not a great film but let …","""POSITIVE""",0.923441
"""ellis_redding_t-800_harry_pott…","""Cheryl Fuller""",false,"""RECOMMENDED ""Deliverance"" with…","""recommended deliverance with a…","""POSITIVE""",0.592365
…,…,…,…,…,…,…
"""katniss_everdeen_superman_harr…","""Bryan Phillips""",true,"""""No one's riding that loco thi…","""no one s riding that loco thin…","""NEGATIVE""",0.831013
"""evoke_wonder_woman_myriad_john…","""Michele Tucker""",true,"""[A Taste of Honey] has an eart…","""has an earthy gusto and sincer…","""POSITIVE""",0.790248
"""miracle_luke_skywalker_destiny…","""William Holland""",true,"""You put up with lines such as …","""you put up with lines such as …","""NEGATIVE""",0.858885
